# Lab 1: Hugging Face Transformers - Getting Started

## 🎯 Learning Objectives

In this lab, you will:
1. Load and use pre-trained transformer models
2. Perform text classification with BERT
3. Generate text with GPT-2
4. Visualize attention patterns
5. Fine-tune a model on custom data

## 📚 Prerequisites
- Basic Python knowledge
- Understanding of transformer architecture (see lecture slides)
- PyTorch basics (helpful but not required)

## ⏱️ Estimated Time: 60-90 minutes

## Part 1: Setup and Installation

In [ ]:
# Install required packages
!pip install transformers datasets torch torchvision torchaudio
!pip install bertviz  # For attention visualization
!pip install scikit-learn matplotlib seaborn

In [ ]:
# Imports
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    pipeline,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Part 2: Using Pre-trained Models with Pipelines

The easiest way to use transformers is through **pipelines**. Let's try different tasks!

### 2.1 Sentiment Analysis

In [ ]:
# Create sentiment analysis pipeline
sentiment_analyzer = pipeline("sentiment-analysis")

# Test examples
texts = [
    "I love this movie! It's absolutely fantastic.",
    "This is the worst product I've ever bought.",
    "The weather is okay, nothing special.",
    "Transformers revolutionized NLP!"
]

results = sentiment_analyzer(texts)

for text, result in zip(texts, results):
    print(f"Text: {text}")
    print(f"Sentiment: {result['label']}, Confidence: {result['score']:.4f}\n")

### 2.2 Text Generation with GPT-2

In [ ]:
# Create text generation pipeline
generator = pipeline("text-generation", model="gpt2")

# Generate text
prompt = "The future of artificial intelligence is"

generated = generator(
    prompt,
    max_length=100,
    num_return_sequences=3,
    temperature=0.8,  # Higher = more creative
    top_p=0.95,  # Nucleus sampling
    do_sample=True
)

print(f"Prompt: {prompt}\n")
for i, gen in enumerate(generated, 1):
    print(f"Generation {i}:")
    print(gen['generated_text'])
    print("-" * 80)

### 2.3 Question Answering

In [ ]:
# Create QA pipeline
qa_pipeline = pipeline("question-answering")

# Context and questions
context = """
The transformer architecture was introduced in the paper 'Attention Is All You Need' 
by Vaswani et al. in 2017. It relies entirely on self-attention mechanisms and 
eliminates recurrence. The model consists of an encoder and decoder, each with 
multiple layers of multi-head attention and feed-forward networks. This architecture 
enabled parallel processing and better handling of long-range dependencies.
"""

questions = [
    "When was the transformer architecture introduced?",
    "What paper introduced transformers?",
    "What does the transformer architecture eliminate?",
    "What are the main components of the transformer?"
]

for question in questions:
    result = qa_pipeline(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {result['answer']} (confidence: {result['score']:.4f})\n")

### 2.4 Named Entity Recognition (NER)

In [ ]:
# Create NER pipeline
ner = pipeline("ner", aggregation_strategy="simple")

text = """
Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 
Cupertino, California on April 1, 1976. The company is now worth over 
$2 trillion and employs more than 150,000 people worldwide.
"""

entities = ner(text)

print("Named Entities:")
for entity in entities:
    print(f"  {entity['word']:20} | {entity['entity_group']:15} | {entity['score']:.4f}")

## Part 3: Deep Dive - Text Classification with BERT

Let's manually work with a BERT model for text classification.

### 3.1 Load Tokenizer and Model

In [ ]:
# Load pre-trained BERT model and tokenizer
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model = model.to(device)

print(f"Model: {model_name}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")

### 3.2 Tokenization Example

In [ ]:
# Example text
text = "Transformers are amazing for NLP tasks!"

# Tokenize
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original text: {text}")
print(f"\nTokens: {tokens}")
print(f"\nToken IDs: {token_ids}")

# Using encode (adds special tokens)
encoded = tokenizer.encode(text, add_special_tokens=True)
print(f"\nEncoded (with special tokens): {encoded}")
print(f"Decoded: {tokenizer.decode(encoded)}")

### 3.3 Inference - Step by Step

In [ ]:
def predict_sentiment(text, model, tokenizer, device):
    """Predict sentiment for a given text"""
    # Tokenize and prepare input
    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    
    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get predictions
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_class].item()
    
    labels = ["NEGATIVE", "POSITIVE"]
    
    return {
        "label": labels[predicted_class],
        "confidence": confidence,
        "probabilities": probabilities[0].cpu().numpy()
    }

# Test
test_texts = [
    "This is absolutely wonderful!",
    "I'm very disappointed with the service.",
    "It's okay, I guess."
]

for text in test_texts:
    result = predict_sentiment(text, model, tokenizer, device)
    print(f"Text: {text}")
    print(f"Prediction: {result['label']} ({result['confidence']:.4f})")
    print(f"Probabilities: NEG={result['probabilities'][0]:.4f}, POS={result['probabilities'][1]:.4f}\n")

## Part 4: Attention Visualization

Let's visualize what the model is paying attention to!

In [ ]:
from bertviz import head_view, model_view
from transformers import AutoModel

# Load model with attention outputs
model_name = "bert-base-uncased"
model = AutoModel.from_pretrained(model_name, output_attentions=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Example sentence
sentence = "The cat sat on the mat because it was comfortable."

# Tokenize
inputs = tokenizer.encode(sentence, return_tensors='pt')
tokens = tokenizer.convert_ids_to_tokens(inputs[0])

# Get attention
with torch.no_grad():
    outputs = model(inputs)
    attention = outputs.attentions  # Tuple of attention weights

print(f"Tokens: {tokens}")
print(f"\nNumber of layers: {len(attention)}")
print(f"Number of heads per layer: {attention[0].shape[1]}")

# Visualize attention (opens in browser)
# Uncomment to run interactively
# head_view(attention, tokens)

### Custom Attention Heatmap

In [ ]:
def plot_attention_heatmap(attention, tokens, layer=0, head=0):
    """Plot attention heatmap for a specific layer and head"""
    # Get attention weights for specific layer and head
    attn = attention[layer][0, head].detach().numpy()
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot heatmap
    sns.heatmap(
        attn,
        xticklabels=tokens,
        yticklabels=tokens,
        cmap='YlOrRd',
        ax=ax,
        cbar_kws={'label': 'Attention Weight'}
    )
    
    ax.set_title(f'Attention Heatmap - Layer {layer}, Head {head}')
    ax.set_xlabel('Key Tokens')
    ax.set_ylabel('Query Tokens')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Visualize different heads
plot_attention_heatmap(attention, tokens, layer=0, head=0)
plot_attention_heatmap(attention, tokens, layer=5, head=3)

## Part 5: Fine-tuning on Custom Dataset

Let's fine-tune a model on the IMDB movie review dataset.

### 5.1 Load Dataset

In [ ]:
# Load IMDB dataset
dataset = load_dataset("imdb")

print("Dataset structure:")
print(dataset)

# Show examples
print("\nExample positive review:")
print(dataset['train'][0]['text'][:200] + "...")
print(f"Label: {dataset['train'][0]['label']}")

# Use small subset for quick training
small_train = dataset['train'].shuffle(seed=42).select(range(1000))
small_test = dataset['test'].shuffle(seed=42).select(range(200))

### 5.2 Prepare Data

In [ ]:
# Load tokenizer and model for fine-tuning
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

# Tokenize datasets
tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)

print("Tokenization complete!")
print(f"Training samples: {len(tokenized_train)}")
print(f"Test samples: {len(tokenized_test)}")

### 5.3 Define Training Configuration

In [ ]:
# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {'accuracy': accuracy}

# Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Trainer ready!")

### 5.4 Train Model

In [ ]:
# Train
print("Starting training...")
trainer.train()
print("Training complete!")

### 5.5 Evaluate Model

In [ ]:
# Evaluate
results = trainer.evaluate()
print("\nEvaluation Results:")
print(f"Accuracy: {results['eval_accuracy']:.4f}")

# Make predictions
predictions = trainer.predict(tokenized_test)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Classification report
print("\nClassification Report:")
print(classification_report(
    true_labels,
    pred_labels,
    target_names=['Negative', 'Positive']
))

### 5.6 Test Fine-tuned Model

In [ ]:
# Test on new examples
test_reviews = [
    "This movie was absolutely brilliant! A masterpiece of cinema.",
    "Terrible acting, poor plot, complete waste of time.",
    "An okay film, nothing special but watchable."
]

print("Fine-tuned Model Predictions:\n")
for review in test_reviews:
    result = predict_sentiment(review, model, tokenizer, device)
    print(f"Review: {review}")
    print(f"Prediction: {result['label']} ({result['confidence']:.4f})\n")

## Part 6: Challenges and Exercises

### Challenge 1: Text Summarization
Use a pre-trained summarization model to summarize long texts.

In [ ]:
# YOUR CODE HERE
# Hint: Use pipeline("summarization", model="facebook/bart-large-cnn")
# Test on a long article or news piece

### Challenge 2: Multi-language Classification
Load a multilingual BERT model and test it on different languages.

In [ ]:
# YOUR CODE HERE
# Hint: Use "bert-base-multilingual-cased"
# Test sentiment on texts in different languages (English, Spanish, French, etc.)

### Challenge 3: Compare Model Sizes
Compare performance and speed between BERT-base and DistilBERT.

In [ ]:
# YOUR CODE HERE
# Load both models, measure inference time and accuracy
# Create comparison visualizations

## 🎓 Summary

In this lab, you learned:
- ✅ How to use Hugging Face transformers with pipelines
- ✅ Different NLP tasks (classification, generation, QA, NER)
- ✅ Manual model loading and inference
- ✅ Attention visualization techniques
- ✅ Fine-tuning transformers on custom data

## 📚 Next Steps
1. Complete the challenges above
2. Explore the [Hugging Face Model Hub](https://huggingface.co/models)
3. Try fine-tuning on your own dataset
4. Move to Lab 2: OLLAMA Setup and Local LLMs

## 💡 Additional Resources
- [Hugging Face Course](https://huggingface.co/course)
- [Transformers Documentation](https://huggingface.co/docs/transformers)
- [BertViz Tutorial](https://github.com/jessevig/bertviz)